# Case study: first SQL questions about the reviews

*Session 3, block 1 · Author: course team · Licence: CC-BY-4.0*

In this notebook you answer first questions about the course data in SQL. We use **DuckDB**, a database
that runs inside the Python process and can query Parquet files directly. Nothing needs to be installed
beyond the course environment, and the SQL is the same as in PostgreSQL for everything we use here.

**Tasks of the practice block**

1. Count the reviews and look at the rating distribution.
2. Compute the average rating per store (JOIN, GROUP BY, HAVING).
3. Join reviews with product prices and count the products without a price (LEFT JOIN, NULL).

Cells marked **TODO** are for you. Solutions are not included; compare your numbers with your neighbours.

In [ ]:
from pathlib import Path

# find the repository root (the folder that contains case-study/), wherever the notebook runs
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir())
DATA = ROOT / "case-study" / "data"
assert (DATA / "train.parquet").exists(), "run case-study/prepare_data.py first"

import duckdb

con = duckdb.connect()  # an in-memory database
# views: names for the Parquet files, so that we can write FROM reviews
con.execute(f"CREATE VIEW reviews  AS SELECT * FROM read_parquet('{DATA / 'train.parquet'}')")
con.execute(f"CREATE VIEW products AS SELECT * FROM read_parquet('{DATA / 'products.parquet'}')")


def q(sql: str):
    """Run a query and return the result as a pandas DataFrame."""
    return con.sql(sql).df()


q("SELECT COUNT(*) AS n_reviews FROM reviews")

## 1. Look at the tables

`DESCRIBE` lists the columns and their types. A **primary key** identifies each row: `review_id` in
`reviews`, `parent_asin` in `products`. `reviews.parent_asin` is a **foreign key**: it refers to a row of
`products`.

In [ ]:
q("DESCRIBE reviews")

In [ ]:
q("DESCRIBE products")

In [ ]:
# is review_id really unique? A primary key must be.
q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT review_id) AS n_ids FROM reviews")

## 2. Filtering and sorting: SELECT, WHERE, ORDER BY, LIMIT

In [ ]:
q("""
SELECT review_id, rating, helpful_vote, LEFT(title, 40) AS title
FROM reviews
WHERE rating = 1 AND verified_purchase      -- both conditions must hold
ORDER BY helpful_vote DESC                  -- most helpful first
LIMIT 5
""")

In [ ]:
# rating distribution
q("""
SELECT rating, COUNT(*) AS n_reviews, ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM reviews
GROUP BY rating
ORDER BY rating
""")

**TODO 1.** How many reviews mention the word *refund* in the title (case-insensitive, `ILIKE '%refund%'`)?
How many 1–2 star reviews were written in 2021 or later?

In [ ]:
# TODO 1: replace ... by a condition, then remove the comment signs
# q("""
# SELECT COUNT(*) AS n
# FROM reviews
# WHERE ...
# """)

## 3. Aggregation: GROUP BY and HAVING

`WHERE` filters rows before grouping; `HAVING` filters groups after aggregation.

In [ ]:
q("""
SELECT parent_asin, COUNT(*) AS n_reviews, ROUND(AVG(rating), 2) AS avg_rating
FROM reviews
GROUP BY parent_asin
HAVING COUNT(*) >= 1000          -- only products with at least 1000 reviews
ORDER BY avg_rating
LIMIT 5
""")

## 4. Average rating per store: INNER JOIN

The store is a product attribute, so we join each review with its product. With an inner join, reviews
without a matching product would disappear; here every review has a product (check it!).

In [ ]:
q("""
SELECT p.store, COUNT(*) AS n_reviews, ROUND(AVG(r.rating), 2) AS avg_rating
FROM reviews AS r
JOIN products AS p ON p.parent_asin = r.parent_asin
GROUP BY p.store
HAVING COUNT(*) >= 1000
ORDER BY avg_rating DESC, n_reviews DESC
LIMIT 10
""")

**TODO 2.** (a) How many groups does the query above have without `LIMIT`? One of them has `store = NULL`:
what does that group contain? (b) Which stores with at least 1000 reviews have the *lowest* average rating?

In [ ]:
# TODO 2

## 5. Products without a price: LEFT JOIN and NULL

`NULL` means *unknown*. Any comparison with NULL is unknown, so `price = NULL` is never true; use
`price IS NULL`. `COUNT(*)` counts rows, `COUNT(price)` counts non-NULL values, and `AVG(price)` ignores
NULLs.

In [ ]:
q("""
SELECT COUNT(*)                 AS n_products,
       COUNT(price)             AS n_with_price,
       COUNT(*) - COUNT(price)  AS n_without_price,
       ROUND(AVG(price), 2)     AS avg_price_of_known
FROM products
""")

In [ ]:
# the classic mistake: = NULL instead of IS NULL
q("SELECT (SELECT COUNT(*) FROM products WHERE price = NULL) AS eq_null, "
  "(SELECT COUNT(*) FROM products WHERE price IS NULL) AS is_null")

In [ ]:
# join reviews with product prices: how many reviews belong to a product without a price?
q("""
SELECT p.price IS NULL AS price_missing,
       COUNT(*)               AS n_reviews,
       COUNT(DISTINCT r.parent_asin) AS n_products,
       ROUND(AVG(r.rating), 2) AS avg_rating
FROM reviews AS r
LEFT JOIN products AS p USING (parent_asin)
GROUP BY price_missing
""")

In [ ]:
# anti-join: products that never received a review in the training period
q("""
SELECT COUNT(*) AS products_without_train_reviews
FROM products AS p
LEFT JOIN reviews AS r ON r.parent_asin = p.parent_asin
WHERE r.review_id IS NULL
""")

**TODO 3.** Among products *with* at least 20 training reviews, what share has no price? Compare with
products with fewer reviews. (Hint: aggregate the reviews per product in a subquery or CTE first, then
join to `products`.) Session 4 returns to this question: is the missing price related to popularity?

In [ ]:
# TODO 3

## 6. The same in PostgreSQL

After block 2 your data are in PostgreSQL. Every query above runs there unchanged; replace the DuckDB
connection by an SQLAlchemy engine:

```python
import os, pandas as pd
from sqlalchemy import create_engine
engine = create_engine(os.environ["DATABASE_URL"])
pd.read_sql("SELECT COUNT(*) FROM reviews", engine)
```